# 1. Импорт библиотек

In [1]:
import os
import sys
sys.path.append('../')

import pandas as pd
pd.set_option('display.max_columns', 450)

import numpy as np

from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

# 2. Загрузка данных

In [2]:
data = pd.read_parquet('../data/processed/data.pqt')

data = data[data['sample_type'].isin(['TRAIN', 'TEST', 'OOT'])]

data.shape

(590540, 633)

In [3]:
valid_features_data = pd.read_excel('../docs/valid_features.xlsx', index_col=False)

In [4]:
FEATURES = valid_features_data.loc[(
    valid_features_data['valid_flag'] == 1)]['attribute'].values

CAT_FEATURES = sorted(set(FEATURES) & set(
    data.select_dtypes(include=["object", "category"]).columns))

NUM_FEATURES = sorted(set(FEATURES) & set(
    data.select_dtypes(include=["number"]).columns))

TRAIN_MASK = (data['sample_type'] == 'TRAIN')
TEST_MASK = (data['sample_type'] == 'TEST')
OOT_MASK = (data['sample_type'] == 'OOT')

In [5]:
STAT_COLS = ['TransactionID', 'target', 'TransactionDT', 'card1']

# финальные категориальные фичи
CAT_FEATURES_FINAL = ['DeviceInfo_brand', 'DeviceType_new',
                      'M1_new', 'M2_new', 'M3_new', 'M4_new', 'M5_new', 'M6_new', 'M7_new', 'M8_new', 'M9_new',
                      'P_emaildomain_grouped', 'ProductCD_new', 'R_emaildomain_grouped',
                      'card4_new', 'card6_new']

NUM_FEATURES_FINAL = [x for x in NUM_FEATURES if x not in ['card1', 'card2', 'card3', 'card5']]

In [6]:
train_df = data[TRAIN_MASK][[*STAT_COLS, *CAT_FEATURES_FINAL, *NUM_FEATURES_FINAL]]

test_df = data[TEST_MASK][[*STAT_COLS, *CAT_FEATURES_FINAL, *NUM_FEATURES_FINAL]]

oot_df = data[OOT_MASK][[*STAT_COLS, *CAT_FEATURES_FINAL, *NUM_FEATURES_FINAL]]

# 3. Преобразование данных

In [7]:
# Категориальные признаки
col_mappings = {}

for col in CAT_FEATURES_FINAL:
    # 
    unique = train_df[col].astype(str).unique()
    mapping = {val: i+1 for i, val in enumerate(unique)}
    # 
    col_mappings[col] = mapping
    
    # 
    train_df[f'{col}_idx'] = train_df[col].astype(str).map(mapping).fillna(0).astype(int)
    test_df[f'{col}_idx'] = test_df[col].astype(str).map(mapping).fillna(0).astype(int)
    oot_df[f'{col}_idx'] = oot_df[col].astype(str).map(mapping).fillna(0).astype(int)
    

#  
train_df.drop(columns=CAT_FEATURES_FINAL, inplace=True)
test_df.drop(columns=CAT_FEATURES_FINAL, inplace=True)
oot_df.drop(columns=CAT_FEATURES_FINAL, inplace=True)

In [8]:
# Числовые
for col in NUM_FEATURES_FINAL:
    train_df[col] = train_df[col].fillna(-999)
    test_df[col] = test_df[col].fillna(-999)
    oot_df[col] = oot_df[col].fillna(-999)

# 
scaler = StandardScaler()

train_df[NUM_FEATURES_FINAL] = scaler.fit_transform(train_df[NUM_FEATURES_FINAL])

test_df[NUM_FEATURES_FINAL] = scaler.transform(test_df[NUM_FEATURES_FINAL])
oot_df[NUM_FEATURES_FINAL] = scaler.transform(oot_df[NUM_FEATURES_FINAL])

# 4. Сохранение данных для collab

In [9]:
path_data = '../data/processed/'

train_df_name = 'train_collab.pqt'
test_df_name = 'test_collab.pqt'
oot_df_name = 'oot_collab.pqt'

train_df.to_parquet(f'{path_data}{train_df_name}')
test_df.to_parquet(f'{path_data}{test_df_name}')
oot_df.to_parquet(f'{path_data}{oot_df_name}')

NameError: name 'train_df' is not defined